<a href="https://colab.research.google.com/github/abdullahh-sheikhh/voxelmorph/blob/dev/scripts/cell_tracking/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell Deformation Estimation with VoxelMorph

2D deformable registration for PhC-C2DH-U373 cells using VoxelMorph, compared against Horn & Schunck classical optical flow baseline.

## 1. Setup

In [ ]:
!git clone https://github.com/abdullahh-sheikhh/voxelmorph.git
%cd voxelmorph
!git checkout dev
!pip install -e . -q
!pip install imagecodecs -q

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Dataset

In [ ]:
from pathlib import Path

!mkdir -p dataset

if not Path('dataset/train/01').exists():
    print('Downloading training data...')
    !wget -q https://data.celltrackingchallenge.net/training-datasets/PhC-C2DH-U373.zip -O dataset/train.zip
    !cd dataset && unzip -q train.zip -d train_raw && mv train_raw/PhC-C2DH-U373/* train/ && rm -rf train_raw train.zip
else:
    print('Training data already exists')

!echo "Train sequences:" && ls dataset/train/01/ | head -3 && echo "... (115 frames total)"

## 3. Visualize Dataset

Show consecutive frame pairs, raw differences, and ground truth cell contours to understand what the model needs to learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from pathlib import Path

# load a few consecutive pairs from sequence 01
data_dir = Path('dataset/train/01')
gt_dir = Path('dataset/train/01_GT/TRA')

pairs_to_show = [(0, 1), (30, 31), (80, 81)]

fig, axes = plt.subplots(len(pairs_to_show), 3, figsize=(15, 5 * len(pairs_to_show)))

for row, (i, j) in enumerate(pairs_to_show):
    src = io.imread(str(data_dir / f't{i:03d}.tif')).astype(np.float32) / 255.0
    tgt = io.imread(str(data_dir / f't{j:03d}.tif')).astype(np.float32) / 255.0
    diff = np.abs(tgt - src)

    # source
    axes[row, 0].imshow(src, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'Frame {i}')
    axes[row, 0].axis('off')

    # try to overlay GT contours
    mask_path = gt_dir / f'man_track{i:03d}.tif'
    if mask_path.exists():
        mask = io.imread(str(mask_path))
        axes[row, 0].contour(mask > 0, colors='cyan', linewidths=0.8, alpha=0.7)

    # target
    axes[row, 1].imshow(tgt, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'Frame {j}')
    axes[row, 1].axis('off')

    mask_path = gt_dir / f'man_track{j:03d}.tif'
    if mask_path.exists():
        mask = io.imread(str(mask_path))
        axes[row, 1].contour(mask > 0, colors='lime', linewidths=0.8, alpha=0.7)

    # raw difference
    im = axes[row, 2].imshow(diff, cmap='inferno', vmin=0, vmax=max(diff.max(), 0.01))
    axes[row, 2].set_title(f'|Frame {j} - Frame {i}|')
    axes[row, 2].axis('off')
    fig.colorbar(im, ax=axes[row, 2], fraction=0.046, pad=0.04)

plt.suptitle('Consecutive Frame Pairs â€” Raw Differences', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Train

Train VoxelMorph (MSE loss, direct displacement) for 100 epochs on both sequences.

In [ ]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 \
    --batch-size 4 \
    --lr 1e-4 \
    --lambda 0.01 \
    --output-dir output \
    --save-every 25

In [ ]:
# display the training loss curve
from IPython.display import Image, display
display(Image(filename='output/loss_curve.png', width=800))

## 5. Evaluate

Compare VoxelMorph against Horn & Schunck optical flow baseline. Compute MSE, Dice (using ground truth segmentation masks), and Jacobian folding percentage.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --output-dir output/eval \
    --max-pairs 0

In [ ]:
# display evaluation visualizations
from IPython.display import Image, display
from pathlib import Path

eval_dir = Path('output/eval')
for img_path in sorted(eval_dir.glob('pair_*.png'))[:5]:
    print(f'\n{img_path.name}')
    display(Image(filename=str(img_path), width=900))

## 6. Model Variant Experiments

Train and compare different configurations from the paper:
- **VM-1**: MSE loss, direct displacement (already trained above)
- **VM-2**: NCC loss, direct displacement
- **VM-3**: MSE loss, diffeomorphic (integration steps = 7)
- **VM-4**: NCC loss, diffeomorphic (integration steps = 7)

In [ ]:
# VM-2: NCC loss, direct displacement
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss ncc --lambda 1.0 --int-steps 0 \
    --output-dir output/vm2_ncc \
    --save-every 25

In [ ]:
# VM-3: MSE loss, diffeomorphic (integration steps = 7)
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss mse --lambda 0.01 --int-steps 7 \
    --output-dir output/vm3_diffeo \
    --save-every 25

In [ ]:
# VM-4: NCC loss, diffeomorphic (integration steps = 7)
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss ncc --lambda 1.0 --int-steps 7 \
    --output-dir output/vm4_diffeo_ncc \
    --save-every 25

In [ ]:
# display all loss curves side by side
from IPython.display import Image, display
from pathlib import Path

for name, output in [('VM-1 (MSE, direct)', 'output'),
                     ('VM-2 (NCC, direct)', 'output/vm2_ncc'),
                     ('VM-3 (MSE, diffeomorphic)', 'output/vm3_diffeo'),
                     ('VM-4 (NCC, diffeomorphic)', 'output/vm4_diffeo_ncc')]:
    curve = Path(output) / 'loss_curve.png'
    if curve.exists():
        print(f'\n{name}')
        display(Image(filename=str(curve), width=700))

In [ ]:
# evaluate all variants (baselines already computed in step 5)
from pathlib import Path

variants = {
    'VM-1 (MSE, direct)': ('output/best.pt', 0, 'output/eval_vm1'),
    'VM-2 (NCC, direct)': ('output/vm2_ncc/best.pt', 0, 'output/eval_vm2'),
    'VM-3 (MSE, diffeomorphic)': ('output/vm3_diffeo/best.pt', 7, 'output/eval_vm3'),
    'VM-4 (NCC, diffeomorphic)': ('output/vm4_diffeo_ncc/best.pt', 7, 'output/eval_vm4'),
}

for name, (model_path, int_steps, eval_dir) in variants.items():
    if not Path(model_path).exists():
        print(f'Skipping {name}: {model_path} not found')
        continue
    print(f'Evaluating {name}...')
    !python -m scripts.cell_tracking.evaluate \
        --model {model_path} \
        --data-dir dataset/train \
        --gt-dir dataset/train \
        --int-steps {int_steps} \
        --output-dir {eval_dir} \
        --no-baselines \
        --max-pairs 0

In [ ]:
# comparison table: baseline + all VoxelMorph variants
import json
from IPython.display import display, Markdown
from pathlib import Path

rows = ['| Method | MSE | Dice | Folding % | Runtime (s/pair) |',
        '|--------|-----|------|-----------|------------------|']

# baseline from the main eval run
bl_path = Path('output/eval') / 'metrics.json'
if bl_path.exists():
    with open(bl_path) as f:
        bm = json.load(f)
    if 'horn_schunck' in bm:
        m = bm['horn_schunck']
        dice = f"{m['dice_mean']:.4f} +/- {m['dice_std']:.4f}" if 'dice_mean' in m else 'N/A'
        rt = f"{m['runtime_mean']:.4f}" if 'runtime_mean' in m else 'N/A'
        rows.append(f"| Horn & Schunck | {m['mse_mean']:.6f} +/- {m['mse_std']:.6f} | {dice} | {m['folding_mean']:.2f}% | {rt} |")

# VoxelMorph variants
for name, edir in [
    ('VM-1 (MSE, direct)', 'output/eval_vm1'),
    ('VM-2 (NCC, direct)', 'output/eval_vm2'),
    ('VM-3 (MSE, diffeomorphic)', 'output/eval_vm3'),
    ('VM-4 (NCC, diffeomorphic)', 'output/eval_vm4'),
]:
    mp = Path(edir) / 'metrics.json'
    if not mp.exists():
        continue
    with open(mp) as f:
        data = json.load(f)
    m = data.get('vxm', data)
    dice = f"{m['dice_mean']:.4f} +/- {m['dice_std']:.4f}" if 'dice_mean' in m else 'N/A'
    rt = f"{m['runtime_mean']:.4f}" if 'runtime_mean' in m else 'N/A'
    rows.append(f"| {name} | {m['mse_mean']:.6f} +/- {m['mse_std']:.6f} | {dice} | {m['folding_mean']:.2f}% | {rt} |")

display(Markdown('\n'.join(rows)))

## 7. Download Results

In [ ]:
# # uncomment to download results
# from google.colab import files
# !zip -r results.zip output/best.pt output/final.pt output/config.json \
#     output/eval/ \
#     output/vm2_ncc/ output/vm3_diffeo/ output/vm4_diffeo_ncc/ \
#     output/eval_vm1/ output/eval_vm2/ output/eval_vm3/ output/eval_vm4/ \
#     2>/dev/null; true
# files.download('results.zip')